<a href="https://colab.research.google.com/github/zshaan25/Sentiment_Analysis_Project/blob/main/Sentiment_Analysis_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sentiment Analysis of Customer Reviews Using Hybrid CNN-LSTM Networks

In [ ]:
import numpy as np
import pandas as pd
import bz2
import gc
import chardet
import re
import os
print(os.listdir("/content/drive/MyDrive/Colab Notebooks/Amazon Polarity"))

['glove.twitter.27B.100d.txt', 'train.txt.txt', 'test.txt.txt']


In [ ]:
from keras.models import Model, Sequential
from keras.layers import Dense, Embedding, Input, Conv1D, GlobalMaxPool1D, Dropout, concatenate, Layer, InputSpec, LSTM
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from keras.callbacks import EarlyStopping, ModelCheckpoint
from keras import backend as K
from keras import activations, initializers, regularizers, constraints
# from keras.utils.conv_utils import conv_output_length
from keras.regularizers import l2
from keras.constraints import MaxNorm

# Read Train & Test Files

In [ ]:
train_file = ('/content/drive/MyDrive/Colab Notebooks/Amazon Polarity/train.txt.txt')
test_file = ('/content/drive/MyDrive/Colab Notebooks/Amazon Polarity/test.txt.txt')

# Create Lists containing Train & Test sentences and convert from raw binary strings to strings that can be parsed

In [ ]:
# Assuming train_file and test_file are strings holding your file paths
with open(train_file, 'r', encoding='utf-8') as f_train:
    train_file_lines = f_train.readlines()

with open(test_file, 'r', encoding='utf-8') as f_test:
    test_file_lines = f_test.readlines()

train_file_lines = train_file_lines[:100000]  # Keeps only the first 100,000 training reviews
test_file_lines = test_file_lines[:20000]    # Keeps only the first 20,000 testing reviews

In [ ]:
del train_file, test_file

In [ ]:
train_labels = [0 if x.split(' ')[0] == '__label__1' else 1 for x in train_file_lines]
train_sentences = [x.split(' ', 1)[1][:-1].lower() for x in train_file_lines]

for i in range(len(train_sentences)):
    train_sentences[i] = re.sub(r'\d','0',train_sentences[i])

test_labels = [0 if x.split(' ')[0] == '__label__1' else 1 for x in test_file_lines]
test_sentences = [x.split(' ', 1)[1][:-1].lower() for x in test_file_lines]

for i in range(len(test_sentences)):
    test_sentences[i] = re.sub(r'\d','0',test_sentences[i])

for i in range(len(train_sentences)):
    if 'www.' in train_sentences[i] or 'http:' in train_sentences[i] or 'https:' in train_sentences[i] or '.com' in train_sentences[i]:
        train_sentences[i] = re.sub(r"([^ ]+(?<=\.[a-z]{3}))", "<url>", train_sentences[i])

for i in range(len(test_sentences)):
    if 'www.' in test_sentences[i] or 'http:' in test_sentences[i] or 'https:' in test_sentences[i] or '.com' in test_sentences[i]:
        test_sentences[i] = re.sub(r"([^ ]+(?<=\.[a-z]{3}))", "<url>", test_sentences[i])

In [ ]:
del train_file_lines, test_file_lines

In [ ]:
gc.collect()

1504

In [ ]:
max_features = 20000
maxlen = 100

In [ ]:
tokenizer = Tokenizer(num_words=max_features)

In [ ]:
tokenizer.fit_on_texts(train_sentences)

In [ ]:
tokenized_train = tokenizer.texts_to_sequences(train_sentences)
X_train = pad_sequences(tokenized_train, maxlen=maxlen)

In [ ]:
X_train[0]

array([    0,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,     0,     0,     0,     0,     0,    75,    11,
           1,   626,  9887,     8,   198,   481,    13,   353,     7,
        6318,     1,    10,    66,   412,    28,    69,     3,    44,
        1640,     7,    75,     5,   129,    68,   634, 13864,   223,
         121,     3,    20,   529,     1,   223,  1734,    17,    43,
           6,    27,     6,     1,   782,     3,    20,   109,   529,
           7,    46,     1,    81,   121,     7,  7007,   256,    38,
        3774,     2,   431,     4, 15311,  1057,    18,  8400,  3167,
           2,  3731,     7,    44,  4671,   194,    68,  2212,     5,
         323], dtype=int32)

In [ ]:
tokenized_test = tokenizer.texts_to_sequences(test_sentences)
X_test = pad_sequences(tokenized_test, maxlen=maxlen)

In [ ]:
EMBEDDING_FILE = '/content/drive/MyDrive/Colab Notebooks/Amazon Polarity/glove.twitter.27B.100d.txt'

In [ ]:
def get_coefs(word, *arr): return word, np.asarray(arr, dtype='float32')
embeddings_index = dict(get_coefs(*o.rstrip().rsplit(' ')) for o in open(EMBEDDING_FILE))

In [ ]:
all_embs = np.stack(list(embeddings_index.values()))
emb_mean,emb_std = all_embs.mean(), all_embs.std()
embed_size = all_embs.shape[1]

word_index = tokenizer.word_index
nb_words = min(max_features, len(word_index))
#change below line if computing normal stats is too slow
embedding_matrix = np.random.normal(emb_mean, emb_std, (nb_words, embed_size)) #embedding_matrix = np.zeros((nb_words, embed_size))
for word, i in word_index.items():
    if i >= max_features: continue
    embedding_vector = embeddings_index.get(word)
    if embedding_vector is not None: embedding_matrix[i] = embedding_vector

In [ ]:
del tokenized_test, tokenized_train, tokenizer, train_sentences, test_sentences, word_index, embeddings_index, all_embs, nb_words
gc.collect()

0

In [ ]:
batch_size = 2048
epochs = 7
embed_size = 100

In [ ]:
gc.collect()

0

In [ ]:
def lstm_model(conv_layers = 2, max_dilation_rate = 3):
    inp = Input(shape=(maxlen, ))
    x = Embedding(max_features, embed_size, weights=[embedding_matrix], trainable=True)(inp)
    x = Dropout(0.25)(x)
    x = Conv1D(2*embed_size, kernel_size = 3)(x)
    prefilt = Conv1D(2*embed_size, kernel_size = 3)(x)
    x = prefilt
    for strides in [1, 1, 2]:
        x = Conv1D(128*2**(strides), strides = strides, kernel_regularizer=l2(4e-6), bias_regularizer=l2(4e-6), kernel_size=3, kernel_constraint=MaxNorm(10), bias_constraint=MaxNorm(10))(x)
    x_f = LSTM(512, kernel_regularizer=l2(4e-6), bias_regularizer=l2(4e-6), kernel_constraint=MaxNorm(10), bias_constraint=MaxNorm(10))(x)
    x_b = LSTM(512, kernel_regularizer=l2(4e-6), bias_regularizer=l2(4e-6), kernel_constraint=MaxNorm(10), bias_constraint=MaxNorm(10))(x)
    x = concatenate([x_f, x_b])
    x = Dropout(0.5)(x)
    x = Dense(64, activation="relu")(x)
    x = Dropout(0.1)(x)
    x = Dense(1, activation="sigmoid")(x)
    model = Model(inputs=inp, outputs=x)
    model.compile(loss='binary_crossentropy',
                  optimizer='adam',
                  metrics=['binary_accuracy'])

    return model

my_model = lstm_model()
my_model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 100)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 100, 100)  │  2,000,000 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 100, 100)  │          0 │ embedding_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_5 (Conv1D)   │ (None, 98, 200)   │     60,200 │ dropout_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_6 (Conv1D)   │ (None, 96, 200)   │    120,200 │ conv1d_5[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_7 (Conv1D)   │ (None, 94, 256)   │    153,856 │ conv1d_6[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_8 (Conv1D)   │ (None, 92, 256)   │    196,864 │ conv1d_7[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_9 (Conv1D)   │ (None, 45, 512)   │    393,728 │ conv1d_8[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_2 (LSTM)       │ (None, 512)       │  2,099,200 │ conv1d_9[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_3 (LSTM)       │ (None, 512)       │  2,099,200 │ conv1d_9[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 1024)      │          0 │ lstm_2[0][0],     │
│ (Concatenate)       │                   │            │ lstm_3[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 1024)      │          0 │ concatenate_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 64)        │     65,600 │ dropout_4[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_5 (Dropout) │ (None, 64)        │          0 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 1)         │         65 │ dropout_5[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 7,188,913 (27.42 MB)

 Trainable params: 7,188,913 (27.42 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
import numpy as np
from keras.callbacks import EarlyStopping, ModelCheckpoint
weight_path="early_weights.keras"
checkpoint = ModelCheckpoint(weight_path, monitor='val_loss', verbose=1, save_best_only=True, mode='min')
early_stopping = EarlyStopping(monitor="val_loss", mode="min", patience=5)
callbacks = [checkpoint, early_stopping]

In [ ]:
# 1. Force convert X_train and train_labels into clean NumPy arrays
# (Assuming X_train has already been passed through pad_sequences earlier)
X_train_fixed = np.array(X_train)
train_labels_fixed = np.array(train_labels)

# 2. Double-check shapes to make sure everything looks right
print("X_train shape:", X_train_fixed.shape)
print("train_labels shape:", train_labels_fixed.shape)

# 3. Start training with the converted arrays
my_model.fit(
    X_train_fixed,
    train_labels_fixed,
    batch_size=batch_size,
    epochs=epochs,
    shuffle=True,
    validation_split=0.20,
    callbacks=callbacks
)

X_train shape: (100000, 100)
train_labels shape: (100000,)
Epoch 1/7
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - binary_accuracy: 0.5116 - loss: 0.8023
Epoch 1: val_loss improved from None to 0.70364, saving model to early_weights.keras

Epoch 1: finished saving model to early_weights.keras
40/40 ━━━━━━━━━━━━━━━━━━━━ 67s 2s/step - binary_accuracy: 0.5122 - loss: 0.7388 - val_binary_accuracy: 0.5126 - val_loss: 0.7036
Epoch 2/7
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - binary_accuracy: 0.5154 - loss: 0.7029
Epoch 2: val_loss improved from 0.70364 to 0.69099, saving model to early_weights.keras

Epoch 2: finished saving model to early_weights.keras
40/40 ━━━━━━━━━━━━━━━━━━━━ 55s 1s/step - binary_accuracy: 0.5137 - loss: 0.7019 - val_binary_accuracy: 0.5126 - val_loss: 0.6910
Epoch 3/7
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - binary_accuracy: 0.5769 - loss: 0.6601
Epoch 3: val_loss improved from 0.69099 to 0.47717, saving model to early_weights.keras

Epoch 3: finished saving model to early_weig

In [ ]:
my_model.load_weights(weight_path)

# Convert test_labels to a numpy array
test_labels_fixed = np.array(test_labels)

score, acc = my_model.evaluate(X_test, test_labels_fixed, batch_size=batch_size)
print('\n--- Final Project Results ---')
print('Test score(loss):', score)
print('Test accuracy:', acc)

10/10 ━━━━━━━━━━━━━━━━━━━━ 5s 501ms/step - binary_accuracy: 0.8946 - loss: 0.2725

--- Final Project Results ---
Test score(loss): 0.272479772567749
Test accuracy: 0.894599974155426


## Final Project Results

The model evaluation showed the following results:

*   **Test score(loss):** 0.2725
*   **Test accuracy:** 0.8946 or 89.46%